# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

---
Queue = week-4 rule score, sorted **within each client**. One reason code and one action per page:
`ZERO_CLICK_YOUNG_PAGE → rewrite_title_and_meta`, `WEAK_CTR_WITH_DEMAND → refresh_snippet_and_intro`,
`YOUNG_PAGE_SETTLING → light_refresh_then_monitor`, `LOW_RISK → monitor`. The table shows the mix in the top 10 per client.

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)

import json
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
SEED = 42
lane["ctr_prev"] = lane.clicks_prev_30d / lane.impressions_prev_30d * 100
for c in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
    lane["log_" + c] = np.log1p(lane[c])
NUM = ["log_impressions_prev_30d", "log_clicks_prev_30d", "log_sessions_prev_30d", "ctr_prev", "content_age_days",
       "search_volume", "competition", "cpc", "word_count"]
X = pd.concat([lane[NUM].fillna(0), lane[NUM].isna().astype(int).add_suffix("_missing").loc[:, ["search_volume_missing", "word_count_missing"]],
               pd.get_dummies(lane[["content_type", "main_intent"]].fillna("unknown"), dtype=int)], axis=1)
y, groups = lane.demand_drop.values, lane.client_id.values
pct = lambda s: s.rank(pct=True)
lane["baseline_score"] = 0.4 * (1 - pct(lane.content_age_days)) + 0.4 * (1 - pct(lane.ctr_prev)) + 0.2 * pct(lane.log_impressions_prev_30d)
def p_at_k(s, l, k):
    return float(np.asarray(l)[np.argsort(-np.asarray(s), kind="stable")[:k]].mean())
rf = lambda: RandomForestClassifier(200, min_samples_leaf=20, random_state=SEED, n_jobs=-1)

def reason(r):
    if r.content_age_days <= 180 and r.clicks_prev_30d == 0: return "ZERO_CLICK_YOUNG_PAGE"
    if r.ctr_prev < 0.5 and r.impressions_prev_30d >= 1000: return "WEAK_CTR_WITH_DEMAND"
    if r.content_age_days <= 180: return "YOUNG_PAGE_SETTLING"
    return "LOW_RISK"
ACTION = {"ZERO_CLICK_YOUNG_PAGE": "rewrite_title_and_meta", "WEAK_CTR_WITH_DEMAND": "refresh_snippet_and_intro",
          "YOUNG_PAGE_SETTLING": "light_refresh_then_monitor", "LOW_RISK": "monitor"}
lane["reason_code"] = lane.apply(reason, axis=1); lane["action"] = lane.reason_code.map(ACTION)
lane["client_rank"] = lane.groupby("client_id").baseline_score.rank(ascending=False, method="first").astype(int)
top = lane[lane.client_rank <= 10]
print(f"top-10 per client: {len(top)} pages, observed drop rate {top.demand_drop.mean():.2f} vs base {lane.demand_drop.mean():.2f}")
top.groupby(["reason_code", "action"]).size().to_frame("pages")

rows x cols: (30000, 44)


top-10 per client: 267 pages, observed drop rate 0.70 vs base 0.62


,,pages
reason_code,action,
LOW_RISK,monitor,63
WEAK_CTR_WITH_DEMAND,refresh_snippet_and_intro,53
YOUNG_PAGE_SETTLING,light_refresh_then_monitor,2
ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,149


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

---
**Who:** a client's content editor, weekly, to choose the first pages to open. **For:** ordering reviews, not
deciding automatically. **Limits:** weak for pages older than a year; blind to client-wide events (migrations,
seasonality); a drop is a proxy for "needs a look", not proof a refresh will help.

In [2]:
lane["age_bucket"] = pd.cut(lane.content_age_days, [89, 180, 365, 5000], labels=["90-180", "181-365", "365+"])
lane.groupby("age_bucket", observed=True).apply(lambda g: pd.Series({"n": len(g),
    "rule_auc": round(roc_auc_score(g.demand_drop, g.baseline_score), 3)}), include_groups=False)

,n,rule_auc
age_bucket,,
90-180,7394.0,0.580
181-365,6404.0,0.585
365+,4212.0,0.544


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

---
**Human checks before acting:** look at the page's current position and queries in Search Console; confirm the
keyword still has demand (skip volume ≈ 0); check no sibling page targets the same topic.
**No-go list:** no client-wide rewrites from this queue; no deleting or redirecting pages; no claims to clients that
a refresh *will* recover traffic; never act on pages with < 100 earlier impressions.

In [3]:
nogo = lane.search_volume.fillna(0) <= 0
print(f"top-10 queue pages flagged for human check (keyword volume 0): {(top.search_volume.fillna(0) <= 0).mean():.0%}")

top-10 queue pages flagged for human check (keyword volume 0): 52%


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

---
Re-check monthly. **Retrain / re-tune triggers:** top-50 precision on the newest month falls below base rate + 0.10;
the base drop rate moves more than 10 points; a new client is > 20% of pages; reason-code mix in the top 50 shifts
sharply. Any of these → re-run W04/W05 notebooks before using the queue again.

In [4]:
monitor = {"precision_at_50_floor": round(float(lane.demand_drop.mean()) + 0.10, 3),
           "current_precision_at_50": p_at_k(lane.baseline_score, y, 50),
           "base_rate": round(float(lane.demand_drop.mean()), 3)}
monitor["status"] = "OK" if monitor["current_precision_at_50"] >= monitor["precision_at_50_floor"] else "RETRAIN"
monitor

{'precision_at_50_floor': 0.716,
 'current_precision_at_50': 0.82,
 'base_rate': 0.616,
 'status': 'OK'}

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

---
The queue CSV is written for the paper (gitignored, regenerated on run); the summary JSON is committed.

In [5]:
OUT = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs"); OUT.mkdir(parents=True, exist_ok=True)
lane.sort_values(["client_id", "client_rank"])[["client_id", "content_id", "client_rank", "baseline_score", "reason_code",
    "action"]].to_csv(OUT / "action_playbook_queue.csv", index=False)
(OUT / "w07_playbook_summary.json").write_text(json.dumps({**monitor, "top10_per_client_pages": int(len(top)),
    "top10_drop_rate": round(float(top.demand_drop.mean()), 3)}, indent=2))
print(sorted(p.name for p in OUT.iterdir()))

['action_playbook_queue.csv', 'baseline_action_score.csv', 'capstone_metrics.json', 'capstone_results.csv', 'figures', 'w04_baseline_metrics.json', 'w05_model_metrics.json', 'w07_playbook_summary.json']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.